# Modelamiento Supervisado

**Caso de Estudio:** Analítica de Clientes — Predicción de Abandono

---

## Descripción

Este notebook entrena todos los modelos supervisados.
Para los modelos de regresión se reportan métricas sobre el conjunto de prueba.
Para los modelos de clasificación se establece una línea base mediante validación cruzada.

---

## Requisitos de Software

- pandas (>=1.1.0)
- numpy (>=2.0.0)
- scikit-learn (>=1.3)

In [1]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [2]:
data = pd.read_csv('../data/dataset_clientes.csv')
data.head()

,id_cliente,fecha_registro,edad,genero,region,estado_civil,ingreso_mensual,gasto_mensual,deuda_total,score_crediticio,...,ultima_compra_dias,uso_app,tipo_plan,num_productos,tiene_tarjeta_credito,canal_registro,dia_semana_registro,hora_registro,codigo_postal,abandono
0,1,2021-10-27,66,Otro,Norte,Divorciado,9.243057e+05,524088.303055,2.448145e+06,455.406680,...,356,Bajo,Estandar,3,1,Tienda,Lunes,22,3824,1
1,2,2018-08-25,51,Masculino,Centro,Soltero,1.384687e+06,314259.751474,1.620569e+06,575.048508,...,307,Medio,Premium,4,1,App,Martes,10,4148,0
2,3,2019-05-25,48,Femenino,Norte,Casado,NaN,387192.316142,5.395040e+06,770.716904,...,232,Alto,Premium,4,1,App,Jueves,6,7200,0
3,4,2022-04-20,54,Masculino,Sur,Casado,4.369032e+05,417328.601856,2.999350e+06,442.722671,...,165,Alto,Estandar,2,1,App,Domingo,16,1782,1
4,5,2020-03-19,31,Otro,Centro,Soltero,7.408561e+05,490961.191253,1.637711e+06,468.188403,...,283,Bajo,Estandar,3,1,Web,Martes,8,3448,1


In [3]:
# Se eliminan los duplicados
data = data.drop_duplicates()

# Fase 3 - Preparación de datos

In [4]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Tratamiento de atípicos via recorte por percentiles.
    """
    def __init__(self, limits=(0.05, 0.05)):
        self.limits = limits

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns_ = X.columns
        else:
            self.columns_ = np.arange(X.shape[1])
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=self.columns_)
        for col in self.columns_:
            lower = X[col].quantile(self.limits[0])
            upper = X[col].quantile(1 - self.limits[1])
            X = X.astype('float64')
            X[col] = np.clip(X[col], lower, upper)
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return np.array(self.columns_)
        return np.array(input_features)

In [5]:
def tratar_duplicados(X: pd.DataFrame, drop: bool = True) -> pd.DataFrame:
    """
    Tratamiento de duplicados.
    Si drop=True elimina filas duplicadas, si no las deja.
    """
    return X.drop_duplicates() if drop else X

In [6]:
class CorrelationFilter(BaseEstimator, TransformerMixin):
    """
    Elimina variables con alta correlación (multicolinealidad).
    """
    def __init__(self, threshold=0.9):
        self.threshold = threshold
        self.columns_to_drop_ = None

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        corr_matrix = X_df.corr().abs()
        upper = corr_matrix.where(
            np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
        )
        self.columns_to_drop_ = [
            col for col in upper.columns if any(upper[col] > self.threshold)
        ]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        return X_df.drop(columns=self.columns_to_drop_, errors='ignore').values

In [7]:
class DataFrameConverter(BaseEstimator, TransformerMixin):
    """
    Convierte el array de ColumnTransformer en DataFrame con nombres de columnas.
    """
    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.feature_names_ = None

    def fit(self, X, y=None):
        self.feature_names_ = self.preprocessor.get_feature_names_out()
        return self

    def transform(self, X):
        return pd.DataFrame(X, columns=self.feature_names_)

# Modelamiento

## Modelos de Regresión

> Se utilizará score_crediticio como target

In [8]:
target_reg = 'score_crediticio'

features_num = [
    'edad', 'ingreso_mensual', 'gasto_mensual', 'deuda_total',
    'antiguedad_meses', 'frecuencia_compra', 'ultima_compra_dias', 'num_productos'
]
features_cat = [
    'genero', 'region', 'estado_civil', 'uso_app', 'tipo_plan', 'canal_registro'
]

X_reg = data[features_num + features_cat]
y_reg = data[target_reg]

mask = y_reg.notna()
X_reg, y_reg = X_reg[mask], y_reg[mask]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=44
)

In [9]:
numeric_transformer = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

### LinearRegression

In [10]:
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_num),
        ('cat', categorical_transformer, features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

modelo_lr = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_lr),
    ('conversion',    DataFrameConverter(preprocessor_lr)),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LinearRegression())
])

modelo_lr.fit(X_train_reg, y_train_reg)
y_pred_lr = modelo_lr.predict(X_test_reg)

r2_lr   = r2_score(y_test_reg, y_pred_lr)
mae_lr  = mean_absolute_error(y_test_reg, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test_reg, y_pred_lr))

print('LinearRegression')
print(f"  R2  : {r2_lr:.4f}")
print(f"  MAE : {mae_lr:,.2f}")
print(f"  RMSE: {rmse_lr:,.2f}")

LinearRegression
  R2  : 0.0009
  MAE : 79.45
  RMSE: 98.92


### DecisionTreeRegressor

In [11]:

modelo_dtr = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('conversion',    DataFrameConverter(preprocessor)),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeRegressor(
                          max_depth=2, min_samples_leaf=200,
                          min_samples_split=100, random_state=44))
])

modelo_dtr.fit(X_train_reg, y_train_reg)
y_pred_dtr = modelo_dtr.predict(X_test_reg)

r2_dtr   = r2_score(y_test_reg, y_pred_dtr)
mae_dtr  = mean_absolute_error(y_test_reg, y_pred_dtr)
rmse_dtr = np.sqrt(mean_squared_error(y_test_reg, y_pred_dtr))

print('DecisionTreeRegressor')
print(f"  R2  : {r2_dtr:.4f}")
print(f"  MAE : {mae_dtr:,.2f}")
print(f"  RMSE: {rmse_dtr:,.2f}")

DecisionTreeRegressor
  R2  : 0.0008
  MAE : 79.45
  RMSE: 98.92


## Modelos de Clasificación

> Se utilizará abandono como target

Se priorizará el estadístico recall esto debido a que queremos priorizar los falsos positivos.

In [12]:
target_cls = 'abandono'

X_cls = data[features_num + features_cat]
y_cls = data[target_cls]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=29, stratify=y_cls
)

### DecisionTreeClassifier

In [13]:
pipeline_dtc = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        DecisionTreeClassifier(random_state=29))
])

cv_dtc = cross_validate(
    pipeline_dtc, X_train_cls, y_train_cls,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

print('DecisionTreeClassifier')
print(f"  Accuracy : {cv_dtc['test_accuracy'].mean():.4f}")
print(f"  F1       : {cv_dtc['test_f1'].mean():.4f}")
print(f"  Precision: {cv_dtc['test_precision'].mean():.4f}")
print(f"  Recall   : {cv_dtc['test_recall'].mean():.4f}")

DecisionTreeClassifier
  Accuracy : 0.5619
  F1       : 0.4539
  Precision: 0.4491
  Recall   : 0.4589


### Logistic Regression

In [14]:
pipeline_logreg = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        LogisticRegression(class_weight='balanced', max_iter=10000, random_state=29))
])

cv_logreg = cross_validate(
    pipeline_logreg, X_train_cls, y_train_cls,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

print('LogisticRegression')
print(f"  Accuracy : {cv_logreg['test_accuracy'].mean():.4f}")
print(f"  F1       : {cv_logreg['test_f1'].mean():.4f}")
print(f"  Precision: {cv_logreg['test_precision'].mean():.4f}")
print(f"  Recall   : {cv_logreg['test_recall'].mean():.4f}")

LogisticRegression
  Accuracy : 0.6255
  F1       : 0.5689
  Precision: 0.5238
  Recall   : 0.6229


### SVM

In [15]:
numeric_transformer_svm = Pipeline(steps=[
    ('winsorizer', Winsorizer()),
    ('imputer',    SimpleImputer(strategy='mean')),
    ('scaler',     StandardScaler())
])

preprocessor_svm = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_svm, features_num),
        ('cat', categorical_transformer,  features_cat)
    ],
    remainder='drop',
    force_int_remainder_cols=False
)

pipeline_svm = Pipeline(steps=[
    ('duplicados',    FunctionTransformer(tratar_duplicados, kw_args={'drop': False})),
    ('preprocesador', preprocessor_svm),
    ('colinealidad',  CorrelationFilter(threshold=0.9)),
    ('modelo',        SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=29))
])

cv_svm = cross_validate(
    pipeline_svm, X_train_cls, y_train_cls,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=29),
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

print('SVM')
print(f"  Accuracy : {cv_svm['test_accuracy'].mean():.4f}")
print(f"  F1       : {cv_svm['test_f1'].mean():.4f}")
print(f"  Precision: {cv_svm['test_precision'].mean():.4f}")
print(f"  Recall   : {cv_svm['test_recall'].mean():.4f}")

SVM
  Accuracy : 0.6137
  F1       : 0.5638
  Precision: 0.5108
  Recall   : 0.6296
